# Day 12, Step 5 — Context Isolation (agent)

**Problem:** Steps 2–4 all build one context for one model call. But when a task splits into independent sub-tasks, giving them one shared context lets details from one leak into (and distract) the other.

**Context isolation** gives each specialist its own separate context and its own model call, so neither sees the other's information. Here the **model triages the ticket first** and decides which specialist(s) are actually needed — `NETWORK`, `ACCOUNT`, or `BOTH` — and LangGraph only runs the branch(es) chosen, with `BOTH` as the safe default when the triage is unclear (better to consult an unnecessary specialist than to skip a needed one). Whichever branch runs, isolation still holds: each specialist only ever sees its own facts.

**Graph:** `START -> triage_ticket -> (agent decision, 1–2 branches) -> build_network_context -> ask_network_specialist -> merge_results` and/or `build_account_context -> ask_account_specialist -> merge_results -> END`

## Setup

In [1]:
from typing import TypedDict

import ollama
from langgraph.graph import END, START, StateGraph

MODEL = "gpt-oss:120b-cloud"


## State

`triage` holds the model's decision. `triage_ticket` also fills all four specialist fields with `(not consulted)` up front, so whichever specialist doesn't run this time still leaves a normal value behind for `merge_results` (and the final printout) to read — no special-cased missing keys to handle.

In [2]:
class State(TypedDict):
    ticket: str
    triage: str
    network_context: str
    network_result: str
    account_context: str
    account_result: str
    merged_context: str
    answer: str


## `triage_ticket` — the agent decision

Asks the model which specialist(s) should look at the ticket. The allowed answers are a fixed set — `NETWORK`, `ACCOUNT`, `BOTH` — and anything else is corrected to `BOTH`: when triage is uncertain, consulting an extra specialist is cheaper than missing one that was needed. It also seeds both specialists' context and result fields with `(not consulted)`, so a branch that doesn't run still leaves a normal value in state.

In [3]:
def triage_ticket(state: State):
    prompt = (
        "A support ticket needs triage. Decide which specialist(s) should look at it.\n"
        "Return only one of: NETWORK, ACCOUNT, BOTH.\n\n"
        f"Ticket: {state['ticket']}"
    )
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    decision = response["message"]["content"].strip().upper()
    if decision not in ("NETWORK", "ACCOUNT", "BOTH"):
        decision = "BOTH"  # safe default: consulting an extra specialist beats missing one
    return {
        "triage": decision,
        "network_context": "(not consulted)",
        "network_result": "(not consulted)",
        "account_context": "(not consulted)",
        "account_result": "(not consulted)",
    }


## `route_after_triage` — where the agent boundary lives

Returns a *list* of node names — LangGraph runs every node named in the list, so `BOTH` fans out to both specialist branches in parallel, while `NETWORK`/`ACCOUNT` runs only one.

In [4]:
def route_after_triage(state: State):
    if state["triage"] == "NETWORK":
        return ["build_network_context"]
    if state["triage"] == "ACCOUNT":
        return ["build_account_context"]
    return ["build_network_context", "build_account_context"]


## The network specialist — isolated context, isolated call

Builds a context containing only the ticket and one network-specific fact, then calls Ollama with only that context — this call never sees anything about the account specialist's world.

In [5]:
def build_network_context(state: State):
    context = (
        f"TASK: {state['ticket']}\n"
        "FACT: The office VPN gateway had a scheduled restart at 09:00 today."
    )
    return {"network_context": context}


def ask_network_specialist(state: State):
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": state['network_context'] + "\nAnswer in one short sentence."}],
    )
    return {"network_result": response["message"]["content"].strip()}


## The account specialist — a separate isolated context, separate call

In [6]:
def build_account_context(state: State):
    context = (
        f"TASK: {state['ticket']}\n"
        "FACT: This user's account was not flagged for any lockouts or password resets today."
    )
    return {"account_context": context}


def ask_account_specialist(state: State):
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": state['account_context'] + "\nAnswer in one short sentence."}],
    )
    return {"account_result": response["message"]["content"].strip()}


## `merge_results` — combine only what actually ran

Reads `network_result` and `account_result` straight from state — whichever specialist the triage skipped still carries the `(not consulted)` default `triage_ticket` set for it. Only the *results* are merged — never the raw isolated contexts.

In [7]:
def merge_results(state: State):
    merged_context = f"NETWORK: {state['network_result']}\nACCOUNT: {state['account_result']}"
    answer = (
        f"Triage: {state['triage']}\n"
        f"Network specialist says: {state['network_result']}\n"
        f"Account specialist says: {state['account_result']}"
    )
    return {"merged_context": merged_context, "answer": answer}


## Build the graph

Both specialist branches point at the same `merge_results` node. LangGraph only waits for the branches that were actually scheduled by `route_after_triage` — if only one specialist ran, `merge_results` runs right after it, without waiting for the other.

In [8]:
graph_builder = StateGraph(State)
graph_builder.add_node("triage_ticket", triage_ticket)
graph_builder.add_node("build_network_context", build_network_context)
graph_builder.add_node("ask_network_specialist", ask_network_specialist)
graph_builder.add_node("build_account_context", build_account_context)
graph_builder.add_node("ask_account_specialist", ask_account_specialist)
graph_builder.add_node("merge_results", merge_results)

graph_builder.add_edge(START, "triage_ticket")
graph_builder.add_conditional_edges("triage_ticket", route_after_triage)
graph_builder.add_edge("build_network_context", "ask_network_specialist")
graph_builder.add_edge("ask_network_specialist", "merge_results")
graph_builder.add_edge("build_account_context", "ask_account_specialist")
graph_builder.add_edge("ask_account_specialist", "merge_results")
graph_builder.add_edge("merge_results", END)
graph = graph_builder.compile()


## Run it

The ticket only mentions a network symptom, so a careful triage should return `NETWORK` and run only that specialist — watch the printed `Triage` line and confirm the account specialist shows `(not consulted)` when that happens. Try a more account-flavored ticket (mentioning a forgotten password) and re-run to see a different triage outcome.

In [9]:
result = graph.invoke({"ticket": "User in Building 3 cannot reach internal systems."})
print("Network specialist saw only:\n", result["network_context"])
print("\nAccount specialist saw only:\n", result["account_context"])
print("\nMerged answer:\n", result["answer"])


Network specialist saw only:
 TASK: User in Building 3 cannot reach internal systems.
FACT: The office VPN gateway had a scheduled restart at 09:00 today.

Account specialist saw only:
 (not consulted)

Merged answer:
 Triage: NETWORK
Network specialist says: The VPN gateway’s scheduled restart at 09:00 likely caused the Building 3 user’s inability to reach internal systems.
Account specialist says: (not consulted)
